In [ ]:
import torch, shutil

assert torch.cuda.is_available(), "Without GPU — activate Runtime > Change runtime type > GPU"

gpu = torch.cuda.get_device_properties(0)
print(f"GPU : {gpu.name}  ({gpu.total_memory/1e9:.1f} GB VRAM)")
print(f"CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

In [ ]:
from google.colab import drive

drive.mount("/content/gdrive")

GDRIVE_ROOT = "/content/gdrive/MyDrive/EK100_MIR"
print(f"Drive mounted: {GDRIVE_ROOT}")

In [ ]:
%%bash

pip install -q pandas numpy tqdm scipy scikit-learn spacy
python -m spacy download en_core_web_sm -q 2>/dev/null || true

echo "Dependencies OK"

In [ ]:
%%bash

cd /content

if [ ! -d "Joint-Part-of-Speech-Embeddings" ]; then
    git clone -q --depth 1 https://github.com/mwray/Joint-Part-of-Speech-Embeddings.git
    echo "Cloned: Joint-Part-of-Speech-Embeddings"
else
    echo "Alredy exist: Joint-Part-of-Speech-Embeddings"
fi

In [ ]:
import subprocess, re, pickle
from pathlib import Path
import numpy as np

ROOT        = "/content"
JPOSE_DIR   = Path(f"{ROOT}/Joint-Part-of-Speech-Embeddings")
GDRIVE_DATA = Path(f"{GDRIVE_ROOT}/data")

In [ ]:
MODELS_DIR     = JPOSE_DIR / "data" / "models"
VID_FEAT_DIR   = JPOSE_DIR / "data" / "video_features"
TXT_FEAT_DIR   = JPOSE_DIR / "data" / "text_features"
DATAFRAMES_DIR = JPOSE_DIR / "data" / "dataframes"
RELATIONAL_DIR = JPOSE_DIR / "data" / "relational"
RELEVANCY_DIR  = JPOSE_DIR / "data" / "relevancy"

In [ ]:
SUBMISSIONS_DIR = Path(f"{GDRIVE_ROOT}/submissions")
ZIPS_DIR        = Path(f"{GDRIVE_ROOT}/submission_zips")
SUBMISSIONS_DIR.mkdir(parents = True, exist_ok = True)
ZIPS_DIR.mkdir(parents = True, exist_ok = True)

In [ ]:
print("Configured paths OK")
print(f"  JPoSE repo : {JPOSE_DIR}")
print(f"  Drive data : {GDRIVE_DATA}")
print(f"  Submissions: {SUBMISSIONS_DIR}")
print(f"  ZIPs       : {ZIPS_DIR}")

In [ ]:
JPOSE_ZIP = GDRIVE_DATA / "JPoSE_data.zip"

In [ ]:
def data_ready():

    checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
    return (
        checkpoint.exists()
        and VID_FEAT_DIR.exists() and any(VID_FEAT_DIR.iterdir())
        and TXT_FEAT_DIR.exists() and any(TXT_FEAT_DIR.iterdir())
    )

In [ ]:
if data_ready():
    print("JPoSE data available")

else:
    assert JPOSE_ZIP.exists(), (
        f"Not found {JPOSE_ZIP}\n"
        "First, run EK100_MIR_submission.ipynb to download and save the data to Drive"
    )

    print(f"Extracting {JPOSE_ZIP.name} ({JPOSE_ZIP.stat().st_size/1e9:.2f} GB)...")

    subprocess.run(
        f'unzip -o -q "{JPOSE_ZIP}" -d "{JPOSE_DIR}"',
        shell = True, check = True,
    )
    print("Complete extraction")

In [ ]:
for label, path in [
    ("models",         MODELS_DIR),
    ("video_features", VID_FEAT_DIR),
    ("text_features",  TXT_FEAT_DIR),
    ("dataframes",     DATAFRAMES_DIR),
    ("relational",     RELATIONAL_DIR),
    ("relevancy",      RELEVANCY_DIR),
]:
    
    files = [f for f in path.rglob("*") if f.is_file()] if path.exists() else []
    status = "OK" if files else "ERR"
    size_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f"  [{status}]  {label}: {len(files)} archivo(s) ({size_mb:.0f} MB)")

checkpoint = MODELS_DIR / "JPoSE_BEST" / "model" / "EPIC_100_retrieval_JPoSE_BEST.pth"
print(f"\n  [{'OK' if checkpoint.exists() else 'ERR'}]  checkpoint: {checkpoint.name}")

In [ ]:
for fpath in JPOSE_DIR.joinpath("src").rglob("*.py"):

    txt = fpath.read_text()
    new = re.sub(
        r"torch\.load\(([^,)]+)\)",
        r"torch.load(\1, weights_only=False)",
        txt,
    )
    if new != txt:
        fpath.write_text(new)
        print(f"  Patched: {fpath.name}")

print("Patches OK")

In [ ]:
MODEL_NAME = "JPoSE_BEST"
COMB_FUNC  = "cat"

CHECKPOINT = MODELS_DIR / MODEL_NAME / "model" / f"EPIC_100_retrieval_{MODEL_NAME}.pth"
assert CHECKPOINT.exists(), f"Checkpoint not found: {CHECKPOINT}"

state = torch.load(str(CHECKPOINT), weights_only = False, map_location = "cpu")
print(f"Checkpoint: {CHECKPOINT.name}")
print(f"Capas ({len(state)} tensores):")
for name, tensor in state.items():
    print(f"  {name:50s}  {str(tuple(tensor.shape)):20s}  {tensor.dtype}")

In [ ]:
jpose_out = SUBMISSIONS_DIR / f"{MODEL_NAME}_test_latest.pkl"

print(f"Running JPoSE inference ({MODEL_NAME}, comb-func = {COMB_FUNC})...")
r = subprocess.run(
    f'cd "{JPOSE_DIR}" && PYTHONPATH="{JPOSE_DIR}/src" '
    f'python src/train/test_jpose_triplet.py "{CHECKPOINT}" '
    f'--comb-func {COMB_FUNC} --challenge-submission "{jpose_out}" 2>&1',
    shell=True, capture_output=True, text=True, timeout=600,
)

print(r.stdout[-2000:])
if r.returncode != 0:
    print("STDERR:", r.stderr[-600:])
    raise RuntimeError("JPoSE inference failed")

In [ ]:
assert jpose_out.exists(), f"Output not generated: {jpose_out}"
with open(jpose_out, "rb") as f:
    sub = pickle.load(f)

sim_mat = np.array(sub["sim_mat"], dtype=np.float32)
vis_ids = list(sub["vis_ids"])
txt_ids = list(sub["txt_ids"])

assert sim_mat.shape == (9668, 3842), f"Shape mismatch: {sim_mat.shape}"
print(f"\nsim_mat : {sim_mat.shape}  dtype = {sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids: {len(txt_ids)}")
print(f"Drive   : {jpose_out.name}  ({jpose_out.stat().st_size/1e6:.1f} MB)")

In [ ]:
SLS_PT = 2
SLS_TL = 3
SLS_TD = 3
SUBMISSION_NAME = f"{MODEL_NAME}_submission"

In [ ]:
def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):

    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   np.array(sim_mat, dtype=np.float32),
        "vis_ids":   [str(v) for v in vis_ids],
        "txt_ids":   [str(t) for t in txt_ids],
    }
    raw = pickle.dumps(payload, protocol=2)
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

tmp_pkl = Path("/tmp/test.pkl")
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
tmp_pkl.write_bytes(pkl_bytes)

In [ ]:
check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == (9668, 3842)
print(f"test.pkl OK: {len(pkl_bytes)/1e6:.1f} MB  |  sim_mat = {np.array(check['sim_mat']).shape}")

zip_path = ZIPS_DIR / f"{SUBMISSION_NAME}.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell = True, check = True, capture_output = True,
)

print(f"ZIP generated : {zip_path.name}  ({zip_path.stat().st_size/1e6:.1f} MB)")
print(f"Drive        : {zip_path}")
print(f"\nReady to upload to: https://www.codabench.org/competitions/12008")